# Sistema Predictivo de Emergencias — Modelo V3-A

**Proyecto Big Data y Analítica de Datos — UPAO**

**Ámbito:** 84 distritos de La Libertad, Perú.

**Objetivo:** priorizar los distritos según la ocurrencia
de al menos una emergencia durante el mes siguiente.

Este notebook documenta el modelo definitivo V3-A,
desarrollado después de revisar las observaciones del
profesor. Los experimentos anteriores se conservan
separadamente como antecedentes.


## 1. Preparación del entorno y datos

Se utiliza Apache Spark y el panel mensual histórico
almacenado en HDFS.


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Proyecto_Emergencias_V3")
    .master("yarn")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "2")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

print("Spark iniciado correctamente")
print("Version:", spark.version)
print("Master:", spark.sparkContext.master)

## 2. Validación de continuidad temporal

Se comprueba la continuidad de los períodos mensuales
por distrito antes de preparar el conjunto de modelado.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

panel_v3 = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "panel_distrito_mes_2003_2025"
)

panel_v3 = panel_v3.withColumn(
    "periodo_idx",
    F.col("anio") * 12 + F.col("mes_num")
)

duplicados = (
    panel_v3.groupBy("ubigeo", "periodo_idx")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

w = Window.partitionBy("ubigeo").orderBy("periodo_idx")

saltos = (
    panel_v3
    .withColumn("periodo_siguiente", F.lead("periodo_idx").over(w))
    .filter(
        F.col("periodo_siguiente").isNotNull() &
        ((F.col("periodo_siguiente") - F.col("periodo_idx")) != 1)
    )
    .count()
)

print("Registros:", panel_v3.count())
print("Distritos:", panel_v3.select("ubigeo").distinct().count())
print("Duplicados:", duplicados)
print("Saltos mensuales:", saltos)
print("Columnas disponibles:", panel_v3.columns)


## 3. Construcción de la variable objetivo

La etiqueta indica si ocurrió al menos una emergencia
durante el mes siguiente. Se utiliza únicamente como
resultado supervisado, no como variable predictora.


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Modelo V3: predicción del mes siguiente
w_v3 = Window.partitionBy("ubigeo").orderBy("periodo_idx")

df_v3 = (
    panel_v3
    .withColumn(
        "periodo_objetivo",
        F.lead("periodo_idx", 1).over(w_v3)
    )
    .withColumn(
        "emergencias_mes_siguiente",
        F.lead("cantidad_emergencias_mes", 1).over(w_v3)
    )
    .withColumn(
        "ocurre_emergencia_mes_siguiente",
        F.when(
            F.col("emergencias_mes_siguiente") > 0, 1.0
        ).when(
            F.col("emergencias_mes_siguiente") == 0, 0.0
        )
    )
    .withColumn(
        "fecha_objetivo",
        F.add_months(
            F.make_date("anio", "mes_num", F.lit(1)),
            1
        )
    )
)

# Conservar únicamente objetivos conocidos y consecutivos
df_modelable_v3 = df_v3.filter(
    (F.col("anio") >= 2004) &
    (F.col("periodo_objetivo") == F.col("periodo_idx") + 1) &
    F.col("ocurre_emergencia_mes_siguiente").isNotNull()
)

print("Filas modelables V3:", df_modelable_v3.count())

print("Distribución de clases:")
df_modelable_v3.groupBy(
    "ocurre_emergencia_mes_siguiente"
).count().show()

print("Ejemplos de predicción:")
df_modelable_v3.select(
    "distrito",
    "anio",
    "mes_num",
    "fecha_objetivo",
    "cantidad_emergencias_mes",
    "emergencias_mes_siguiente",
    "ocurre_emergencia_mes_siguiente"
).orderBy("anio", "mes_num", "distrito").show(10, truncate=False)


## 4. Construcción de las 19 variables predictoras

Se utilizan registros mensuales de emergencias,
impactos, precipitaciones y estacionalidad.
Las ventanas históricas excluyen el mes actual.


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import math

# Ventanas móviles por distrito
w_v3 = Window.partitionBy("ubigeo").orderBy("periodo_idx")
w3_v3 = w_v3.rowsBetween(-3, -1)
w6_v3 = w_v3.rowsBetween(-6, -1)
w12_v3 = w_v3.rowsBetween(-12, -1)

# Trabajamos sobre el panel completo para conservar el historial
df_features_v3 = df_v3

# Emergencias históricas
df_features_v3 = (
    df_features_v3
    .withColumn("emergencias_mes_anterior",
                F.lag("cantidad_emergencias_mes", 1).over(w_v3))
    .withColumn("emergencias_3_meses",
                F.sum("cantidad_emergencias_mes").over(w3_v3))
    .withColumn("emergencias_6_meses",
                F.sum("cantidad_emergencias_mes").over(w6_v3))
    .withColumn("emergencias_12_meses",
                F.sum("cantidad_emergencias_mes").over(w12_v3))
    .withColumn("emergencias_mismo_mes_anio_anterior",
                F.lag("cantidad_emergencias_mes", 12).over(w_v3))
)

# Impacto histórico
for origen, destino in [
    ("afectados_mes", "afectados_12_meses"),
    ("damnificados_mes", "damnificados_12_meses"),
    ("fallecidos_mes", "fallecidos_12_meses"),
    ("viviendas_afectadas_mes", "viviendas_afectadas_12_meses"),
    ("viviendas_destruidas_mes", "viviendas_destruidas_12_meses")
]:
    df_features_v3 = df_features_v3.withColumn(
        destino, F.sum(origen).over(w12_v3)
    )

# Precipitación histórica
df_features_v3 = (
    df_features_v3
    .withColumn("lluvia_mes_anterior",
                F.lag("precipitacion_total_mm", 1).over(w_v3))
    .withColumn("lluvia_3_meses",
                F.sum("precipitacion_total_mm").over(w3_v3))
)

# Estacionalidad
df_features_v3 = (
    df_features_v3
    .withColumn("mes_sin",
                F.sin(2 * F.lit(math.pi) * F.col("mes_num") / 12))
    .withColumn("mes_cos",
                F.cos(2 * F.lit(math.pi) * F.col("mes_num") / 12))
)

# Las 19 variables originales
features_v3_19 = [
    "cantidad_emergencias_mes",
    "tipos_peligro_mes",
    "emergencias_mes_anterior",
    "emergencias_3_meses",
    "emergencias_6_meses",
    "emergencias_12_meses",
    "emergencias_mismo_mes_anio_anterior",
    "afectados_12_meses",
    "damnificados_12_meses",
    "fallecidos_12_meses",
    "viviendas_afectadas_12_meses",
    "viviendas_destruidas_12_meses",
    "precipitacion_total_mm",
    "precipitacion_promedio_diaria_mm",
    "precipitacion_maxima_diaria_mm",
    "lluvia_mes_anterior",
    "lluvia_3_meses",
    "mes_sin",
    "mes_cos"
]

# Filtrar después de calcular las ventanas históricas
df_modelable_v3 = df_features_v3.filter(
    (F.col("anio") >= 2004) &
    (F.col("periodo_objetivo") == F.col("periodo_idx") + 1) &
    F.col("ocurre_emergencia_mes_siguiente").isNotNull()
)

# Verificaciones
nulos = df_modelable_v3.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in features_v3_19
]).first().asDict()

print("Registros V3:", df_modelable_v3.count())
print("Variables V3-A:", len(features_v3_19))

print("Variables con nulos:",
      {k: v for k, v in nulos.items() if v > 0})


## 5. División temporal de los datos

La separación se realiza según el año del mes objetivo:

- Entrenamiento inicial: hasta 2023.
- Validación: 2024.
- Prueba independiente: 2025.

Esto permite seleccionar el modelo antes de evaluar
su comportamiento en el período de prueba.


In [ ]:

from pyspark.sql import functions as F

# Separación según la fecha del MES OBJETIVO
df_v3_temporal = df_modelable_v3.withColumn(
    "anio_objetivo",
    F.year("fecha_objetivo")
)

# Entrenamiento: resultados hasta 2023
train_v3 = df_v3_temporal.filter(
    F.col("anio_objetivo") <= 2023
)

# Validación: resultados de 2024
val_v3 = df_v3_temporal.filter(
    F.col("anio_objetivo") == 2024
)

# Prueba: resultados de 2025
test_v3 = df_v3_temporal.filter(
    F.col("anio_objetivo") == 2025
)

print("ENTRENAMIENTO:", train_v3.count())
print("VALIDACION:", val_v3.count())
print("PRUEBA:", test_v3.count())

print("\nDistribución de clases:")

for nombre, df in [
    ("TRAIN", train_v3),
    ("VALIDACION", val_v3),
    ("TEST", test_v3)
]:
    print("\n", nombre)
    df.groupBy(
        "ocurre_emergencia_mes_siguiente"
    ).count().orderBy(
        "ocurre_emergencia_mes_siguiente"
    ).show()



## 6. Entrenamiento y validación inicial V3-A

Regresión logística ponderada con 19 variables.
Los pesos se calculan exclusivamente con el conjunto
de entrenamiento.


In [ ]:

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

label_v3 = "ocurre_emergencia_mes_siguiente"

# 1. Calcular pesos únicamente con el entrenamiento
conteos = {
    float(r[label_v3]): r["count"]
    for r in train_v3.groupBy(label_v3).count().collect()
}

n = sum(conteos.values())
peso_no = n / (2 * conteos[0.0])
peso_si = n / (2 * conteos[1.0])

print("Peso NO:", round(peso_no, 4))
print("Peso SI:", round(peso_si, 4))

train_v3_pesos = train_v3.withColumn(
    "peso_clase",
    F.when(F.col(label_v3) == 1.0, F.lit(peso_si))
     .otherwise(F.lit(peso_no))
)

# 2. Vectorizar las 19 variables
assembler_v3a = VectorAssembler(
    inputCols=features_v3_19,
    outputCol="features",
    handleInvalid="error"
)

train_v3a_vec = assembler_v3a.transform(train_v3_pesos)
val_v3a_vec = assembler_v3a.transform(val_v3)

# 3. Entrenar regresión logística ponderada
lr_v3a = LogisticRegression(
    labelCol=label_v3,
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_v3a = lr_v3a.fit(train_v3a_vec)

print("Modelo V3-A entrenado correctamente")

# 4. Evaluar únicamente con validación 2024
pred_v3a = modelo_v3a.transform(val_v3a_vec)

matriz = {
    (float(r[label_v3]), float(r["prediction"])): r["count"]
    for r in pred_v3a.groupBy(
        label_v3, "prediction"
    ).count().collect()
}

tp = matriz.get((1.0, 1.0), 0)
fn = matriz.get((1.0, 0.0), 0)
fp = matriz.get((0.0, 1.0), 0)
tn = matriz.get((0.0, 0.0), 0)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

f2 = (
    5 * precision * recall / (4 * precision + recall)
    if 4 * precision + recall else 0
)

evaluador = BinaryClassificationEvaluator(
    labelCol=label_v3,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

pr_auc = evaluador.evaluate(pred_v3a)

print("\n===== VALIDACIÓN V3-A: 2024 =====")
print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))
print("F2:", round(f2, 4))
print("PR-AUC:", round(pr_auc, 4))


## 7. Entrenamiento definitivo V3-A

Después de seleccionar la configuración V3-A, se
entrena el modelo definitivo con meses objetivo hasta
2024, reservando 2025 para la evaluación final.


In [ ]:

from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Modelo seleccionado: V3-A, 19 variables, horizonte +1
label_v3 = "ocurre_emergencia_mes_siguiente"

# División temporal final según el MES OBJETIVO
train_final_v3 = df_v3_temporal.filter(
    F.col("anio_objetivo") <= 2024
)

test_final_v3 = df_v3_temporal.filter(
    F.col("anio_objetivo") == 2025
)

print("Train final:", train_final_v3.count())
print("Test final:", test_final_v3.count())

# Calcular pesos solo con el entrenamiento final
conteos_final = {
    float(r[label_v3]): r["count"]
    for r in train_final_v3.groupBy(label_v3).count().collect()
}

n_final = sum(conteos_final.values())

peso_no_final = n_final / (2 * conteos_final[0.0])
peso_si_final = n_final / (2 * conteos_final[1.0])

print("Peso NO:", round(peso_no_final, 4))
print("Peso SI:", round(peso_si_final, 4))

train_final_v3 = train_final_v3.withColumn(
    "peso_clase",
    F.when(
        F.col(label_v3) == 1.0,
        F.lit(peso_si_final)
    ).otherwise(F.lit(peso_no_final))
)

# Vectorizar las mismas 19 variables
assembler_final_v3 = VectorAssembler(
    inputCols=features_v3_19,
    outputCol="features",
    handleInvalid="error"
)

train_final_vec = assembler_final_v3.transform(
    train_final_v3
)

test_final_vec = assembler_final_v3.transform(
    test_final_v3
)

# Reentrenamiento final
lr_final_v3 = LogisticRegression(
    labelCol=label_v3,
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_final_v3 = lr_final_v3.fit(train_final_vec)

# Predicciones de 2025
pred_final_v3 = modelo_final_v3.transform(
    test_final_vec
)

print("V3-A final entrenada correctamente")
print("Predicciones generadas:", pred_final_v3.count())


## 8. Persistencia del modelo en HDFS

Se guarda el modelo entrenado y el VectorAssembler
para reutilizarlos desde el motor predictivo.


In [ ]:

# Guardar V3-A sin sobrescribir los modelos anteriores

ruta_modelo_v3 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/models/"
    "lr_distrito_mes_v3_1mes_19vars"
)

modelo_final_v3.write().save(ruta_modelo_v3)

print("Modelo V3-A guardado correctamente")
print("Ruta:", ruta_modelo_v3)
print("Variables:", len(features_v3_19))
print("Horizonte: 1 mes")


In [ ]:

# PASO 111 - Guardar VectorAssembler V3-A

ruta_assembler = (
    "hdfs://localhost:9000/proyecto_emergencias/"
    "models/assembler_v3_1mes_19vars"
)

# Recuperar las variables del ensamblador
variables = assembler_final_v3.getInputCols()

print("===== COMPONENTES DEL MODELO V3-A =====")
print("Variables utilizadas:", len(variables))

for i, variable in enumerate(variables, start=1):
    print(f"{i:02d}. {variable}")

# Validaciones
assert len(variables) == 19
assert len(set(variables)) == 19

assert (
    assembler_final_v3.getOutputCol()
    == modelo_v3_demo.getFeaturesCol()
)

# Guardar sin sobrescribir componentes existentes
assembler_final_v3.write().save(ruta_assembler)

print("\nVectorAssembler guardado en:")
print(ruta_assembler)

print("\nPASO 111 COMPLETADO")



## 9. Evaluación retrospectiva de V3-A

Se evalúa la priorización territorial de los 84 distritos
mediante Recall@20 y Precision@20.

Se comparan los 20 primeros distritos del modelo V3-A
con los 20 primeros del ranking histórico basado en
emergencias acumuladas durante 12 meses.

Los CSV conservan las predicciones y las etiquetas reales
para octubre, noviembre y diciembre de 2025.

Las puntuaciones del modelo no deben interpretarse
como probabilidades calibradas.


In [ ]:

# EVALUACION_REPRODUCIBLE_V3A
from pathlib import Path
import pandas as pd

carpetas = [
    Path("data/evaluaciones_2025"),
    Path("../data/evaluaciones_2025")
]

carpeta = next(
    (p for p in carpetas if p.is_dir()),
    None
)

if carpeta is None:
    raise FileNotFoundError(
        "No se encontró data/evaluaciones_2025"
    )

archivos = {
    "Octubre 2025": "prediccion_octubre_2025.csv",
    "Noviembre 2025": "prediccion_noviembre_2025.csv",
    "Diciembre 2025": "prediccion_diciembre_2025.csv"
}

resultados = []

for mes, archivo in archivos.items():

    df = pd.read_csv(
        carpeta / archivo,
        dtype={"ubigeo": str}
    )

    requeridas = {
        "ubigeo",
        "puesto_modelo",
        "puesto_historico",
        "ocurrio_realmente"
    }

    if not requeridas.issubset(df.columns):
        raise ValueError(f"Columnas incompletas: {mes}")

    if len(df) != 84 or df["ubigeo"].nunique() != 84:
        raise ValueError(f"84 distritos esperados: {mes}")

    if df[list(requeridas)].isna().any().any():
        raise ValueError(f"Datos faltantes: {mes}")

    df["ocurrio_realmente"] = pd.to_numeric(
        df["ocurrio_realmente"]
    )

    if not df["ocurrio_realmente"].isin([0, 1]).all():
        raise ValueError(f"Etiquetas inválidas: {mes}")

    for columna in ["puesto_modelo", "puesto_historico"]:
        df[columna] = pd.to_numeric(df[columna])

        if sorted(df[columna].tolist()) != list(range(1, 85)):
            raise ValueError(f"Ranking inválido: {mes}, {columna}")

    reales = int(df["ocurrio_realmente"].sum())

    if reales == 0:
        raise ValueError(f"Sin positivos evaluables: {mes}")

    aciertos_v3 = int(
        df.nsmallest(20, "puesto_modelo")
        ["ocurrio_realmente"].sum()
    )

    aciertos_hist = int(
        df.nsmallest(20, "puesto_historico")
        ["ocurrio_realmente"].sum()
    )

    resultados.append({
        "Mes": mes,
        "Positivos reales": reales,
        "Aciertos V3-A": aciertos_v3,
        "Aciertos histórico": aciertos_hist,
        "Recall@20 V3-A (%)": round(
            100 * aciertos_v3 / reales, 2
        ),
        "Recall@20 histórico (%)": round(
            100 * aciertos_hist / reales, 2
        ),
        "Precision@20 V3-A (%)": round(
            100 * aciertos_v3 / 20, 2
        )
    })

evaluacion_v3a = pd.DataFrame(resultados)

print("===== EVALUACION RETROSPECTIVA V3-A =====")
print(evaluacion_v3a.to_string(index=False))
